# Similaridade entre topicos da pauta e do discurso
Este notebook gera visualizacoes comparando a similaridade entre topicos da pauta e topicos do discurso antes e depois da eleicao, por partido, para os fluxos v1 e v2.

In [14]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / "data" / "party_agenda" / "embeddings"
PARTIES = ["MDB", "NOVO", "PL", "PSOL", "PT", "UNIAO"]
FLOWS = ["v1", "v2"]
ELECTION_PERIODS = ["antesDaEleicao", "depoisDaEleicao"]
CSV_NAME = "similaridade_topics_discurso_topics_agenda.csv"

def load_similarity_table(party: str, flow: str, period: str) -> pd.DataFrame:
    file_path = DATA_DIR / party / "similaridade" / "topics" / "lda" / flow / period / CSV_NAME
    if not file_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(file_path)
    df["party"] = party
    df["flow"] = flow
    df["period"] = period
    return df

rows = []
for party in PARTIES:
    for flow in FLOWS:
        for period in ELECTION_PERIODS:
            rows.append(load_similarity_table(party, flow, period))

similarity_df = pd.concat([df for df in rows if not df.empty], ignore_index=True)
similarity_df.head()

,agenda_topic,agenda_terms,discourse_topic,discourse_terms,cosine_similarity,party,flow,period
0,0,"0.012*""brasil"" + 0.008*""social"" + 0.008*""encon...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.624478,MDB,v1,antesDaEleicao
1,1,"0.017*""brasil"" + 0.014*""país"" + 0.009*""público...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.704395,MDB,v1,antesDaEleicao
2,2,"0.009*""brasil"" + 0.007*""nacional"" + 0.007*""paí...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.716476,MDB,v1,antesDaEleicao
3,0,"0.012*""brasil"" + 0.008*""social"" + 0.008*""encon...",4,"0.009*""real"" + 0.008*""povo"" + 0.008*""votar"" + ...",0.607640,MDB,v1,depoisDaEleicao
4,1,"0.017*""brasil"" + 0.014*""país"" + 0.009*""público...",4,"0.009*""real"" + 0.008*""povo"" + 0.008*""votar"" + ...",0.563006,MDB,v1,depoisDaEleicao


In [15]:
if similarity_df.empty:
    raise ValueError("Nenhum arquivo foi encontrado. Verifique os caminhos dos CSVs.")

similarity_df["cosine_similarity"] = pd.to_numeric(similarity_df["cosine_similarity"], errors="coerce")
summary = (
    similarity_df
    .groupby(["party", "flow", "period"], as_index=False)
    .agg(mean_similarity=("cosine_similarity", "mean"),
         median_similarity=("cosine_similarity", "median"),
         max_similarity=("cosine_similarity", "max"),
         rows=("cosine_similarity", "size"))
)
summary

,party,flow,period,mean_similarity,median_similarity,max_similarity,rows
0,MDB,v1,antesDaEleicao,0.681783,0.704395,0.716476,3
1,MDB,v1,depoisDaEleicao,0.591585,0.604109,0.607640,3
2,MDB,v2,antesDaEleicao,0.364040,0.355441,0.650195,7
3,MDB,v2,depoisDaEleicao,0.444032,0.456700,0.744597,7
4,NOVO,v1,antesDaEleicao,0.717272,0.673493,0.837653,3
5,NOVO,v1,depoisDaEleicao,0.637633,0.583809,0.773036,3
6,NOVO,v2,antesDaEleicao,0.609671,0.618455,0.753739,4
7,NOVO,v2,depoisDaEleicao,0.549626,0.565147,0.698845,4
8,PL,v1,antesDaEleicao,0.545564,0.566321,0.681587,6
9,PL,v1,depoisDaEleicao,0.491892,0.427199,0.730019,6


In [16]:
fig_v1 = px.bar(
    summary[summary["flow"] == "v1"],
    x="party",
    y="mean_similarity",
    color="period",
    barmode="group",
    title="Similaridade media entre topicos de pauta e discurso (v1)",
    labels={"mean_similarity": "Similaridade media", "period": "Periodo"},
    text=summary.loc[summary["flow"] == "v1", "mean_similarity"].round(3).astype(str),
)
fig_v1.update_layout(height=420, legend_title_text="Periodo")
fig_v1.update_traces(textposition="outside")
fig_v1.show()

In [17]:
fig_v2 = px.bar(
    summary[summary["flow"] == "v2"],
    x="party",
    y="mean_similarity",
    color="period",
    barmode="group",
    title="Similaridade media entre topicos de pauta e discurso (v2)",
    labels={"mean_similarity": "Similaridade media", "period": "Periodo"},
    text=summary.loc[summary["flow"] == "v2", "mean_similarity"].round(3).astype(str),
)
fig_v2.update_layout(height=420, legend_title_text="Periodo")
fig_v2.update_traces(textposition="outside")
fig_v2.show()

In [18]:
# Tabela comparativa dos topicos mais similares
top_matches = (
    similarity_df
    .sort_values(["party", "flow", "period", "cosine_similarity"], ascending=[True, True, True, False])
    .groupby(["party", "flow", "period"], as_index=False)
    .head(5)
    .reset_index(drop=True)
)
top_matches.head()

,agenda_topic,agenda_terms,discourse_topic,discourse_terms,cosine_similarity,party,flow,period
0,2,"0.009*""brasil"" + 0.007*""nacional"" + 0.007*""paí...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.716476,MDB,v1,antesDaEleicao
1,1,"0.017*""brasil"" + 0.014*""país"" + 0.009*""público...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.704395,MDB,v1,antesDaEleicao
2,0,"0.012*""brasil"" + 0.008*""social"" + 0.008*""encon...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.624478,MDB,v1,antesDaEleicao
3,0,"0.012*""brasil"" + 0.008*""social"" + 0.008*""encon...",4,"0.009*""real"" + 0.008*""povo"" + 0.008*""votar"" + ...",0.607640,MDB,v1,depoisDaEleicao
4,2,"0.009*""brasil"" + 0.007*""nacional"" + 0.007*""paí...",4,"0.009*""real"" + 0.008*""povo"" + 0.008*""votar"" + ...",0.604109,MDB,v1,depoisDaEleicao


In [19]:
fig = px.scatter(
    top_matches,
    x="agenda_topic",
    y="discourse_topic",
    size="cosine_similarity",
    color="period",
    facet_row="party",
    facet_col="flow",
    title="Topicos mais similares entre pauta e discurso (top 5 por partido/fluxo/periodo)",
    labels={"agenda_topic": "Topico de pauta", "discourse_topic": "Topico de discurso"},
    hover_data=["cosine_similarity", "agenda_terms", "discourse_terms"],
)
fig.update_layout(height=220 * len(PARTIES))
fig.show()

In [20]:
# Topicos do UNIAO - v1 (antes e depois)
uniao_v1 = (
    similarity_df[(similarity_df["party"] == "UNIAO") & (similarity_df["flow"] == "v1")]
    .sort_values(["period", "cosine_similarity"], ascending=[True, False])
    [["period", "agenda_topic", "agenda_terms", "discourse_topic", "discourse_terms", "cosine_similarity"]]
)
if uniao_v1.empty:
    raise ValueError("Sem dados para UNIAO no fluxo v1.")

uniao_v1_display = uniao_v1.copy()
uniao_v1_display["cosine_similarity"] = uniao_v1_display["cosine_similarity"].round(3)

fig_uniao_v1 = go.Figure(
    data=[
        go.Table(
            header=dict(values=list(uniao_v1_display.columns), fill_color="#e9ecef", align="left"),
            cells=dict(values=[uniao_v1_display[col] for col in uniao_v1_display.columns], align="left"),
        )
    ]
)
fig_uniao_v1.update_layout(title="Topicos da pauta e do discurso do UNIAO (v1)", height=520)
fig_uniao_v1.show()

In [21]:
# Topicos do UNIAO - v2 (antes e depois)
uniao_v2 = (
    similarity_df[(similarity_df["party"] == "UNIAO") & (similarity_df["flow"] == "v2")]
    .sort_values(["period", "cosine_similarity"], ascending=[True, False])
    [["period", "agenda_topic", "agenda_terms", "discourse_topic", "discourse_terms", "cosine_similarity"]]
)
if uniao_v2.empty:
    raise ValueError("Sem dados para UNIAO no fluxo v2.")

uniao_v2_display = uniao_v2.copy()
uniao_v2_display["cosine_similarity"] = uniao_v2_display["cosine_similarity"].round(3)

fig_uniao_v2 = go.Figure(
    data=[
        go.Table(
            header=dict(values=list(uniao_v2_display.columns), fill_color="#e9ecef", align="left"),
            cells=dict(values=[uniao_v2_display[col] for col in uniao_v2_display.columns], align="left"),
        )
    ]
)
fig_uniao_v2.update_layout(title="Topicos da pauta e do discurso do UNIAO (v2)", height=520)
fig_uniao_v2.show()

In [22]:
# Tabela com topicos mais similares para todos os partidos (v1 e v2)

def render_topic_table(df: pd.DataFrame, title: str) -> None:
    if df.empty:
        raise ValueError(f"Sem dados para {title}.")
    df = df.copy()
    df["cosine_similarity"] = df["cosine_similarity"].round(3)
    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=list(df.columns), fill_color="#e9ecef", align="left"),
                cells=dict(values=[df[col] for col in df.columns], align="left"),
            )
        ]
    )
    fig.update_layout(title=title, height=420)
    fig.show()

all_v1 = (
    top_matches[top_matches["flow"] == "v1"]
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    [["party", "period", "agenda_topic", "discourse_topic", "cosine_similarity"]]
)
all_v2 = (
    top_matches[top_matches["flow"] == "v2"]
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    [["party", "period", "agenda_topic", "discourse_topic", "cosine_similarity"]]
)

render_topic_table(all_v1, "Topicos mais similares por partido (v1)")
render_topic_table(all_v2, "Topicos mais similares por partido (v2)")

In [23]:
# Tabela resumida para exportacao ou inspecao
summary_table = summary.sort_values(["party", "flow", "period"])
summary_table

,party,flow,period,mean_similarity,median_similarity,max_similarity,rows
0,MDB,v1,antesDaEleicao,0.681783,0.704395,0.716476,3
1,MDB,v1,depoisDaEleicao,0.591585,0.604109,0.607640,3
2,MDB,v2,antesDaEleicao,0.364040,0.355441,0.650195,7
3,MDB,v2,depoisDaEleicao,0.444032,0.456700,0.744597,7
4,NOVO,v1,antesDaEleicao,0.717272,0.673493,0.837653,3
5,NOVO,v1,depoisDaEleicao,0.637633,0.583809,0.773036,3
6,NOVO,v2,antesDaEleicao,0.609671,0.618455,0.753739,4
7,NOVO,v2,depoisDaEleicao,0.549626,0.565147,0.698845,4
8,PL,v1,antesDaEleicao,0.545564,0.566321,0.681587,6
9,PL,v1,depoisDaEleicao,0.491892,0.427199,0.730019,6
